# SBA Loan Analysis
Jay Karki | [LinkedIn](https://www.linkedin.com/in/jaykarki/)

Cleaning and preparing ~899K SBA loan records to analyze what predicts loan default.

Input: `data/raw/SBAnational.csv`  
Output: `data/clean/final_sba_data.csv`

In [21]:
import pandas as pd
import numpy as np


## Data Ingestion
Loading the raw SBA national loan dataset. Confirming shape and previewing the first few rows before touching anything.

In [22]:
df = pd.read_csv('../data/raw/SBAnational.csv')

print(df.shape)
print(df.head())

/var/folders/f2/53nz7krx6_l_ft8vfq303_lm0000gn/T/ipykernel_67105/4279793673.py:1: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/raw/SBAnational.csv')


(899164, 27)
   LoanNr_ChkDgt                           Name          City State    Zip  \
0     1000014003                 ABC HOBBYCRAFT    EVANSVILLE    IN  47711   
1     1000024006    LANDMARK BAR & GRILLE (THE)     NEW PARIS    IN  46526   
2     1000034009          WHITLOCK DDS, TODD M.   BLOOMINGTON    IN  47401   
3     1000044001  BIG BUCKS PAWN & JEWELRY, LLC  BROKEN ARROW    OK  74012   
4     1000054004    ANASTASIA CONFECTIONS, INC.       ORLANDO    FL  32801   

                            Bank BankState   NAICS ApprovalDate ApprovalFY  \
0               FIFTH THIRD BANK        OH  451120    28-Feb-97       1997   
1                1ST SOURCE BANK        IN  722410    28-Feb-97       1997   
2        GRANT COUNTY STATE BANK        IN  621210    28-Feb-97       1997   
3  1ST NATL BK & TR CO OF BROKEN        OK       0    28-Feb-97       1997   
4        FLORIDA BUS. DEVEL CORP        FL       0    28-Feb-97       1997   

   ...  RevLineCr  LowDoc  ChgOffDate  Disburseme

## Exploratory Data Analysis
Checking column types and null counts across all 27 columns before making any changes.

In [23]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 899164 entries, 0 to 899163
Data columns (total 27 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   LoanNr_ChkDgt      899164 non-null  int64  
 1   Name               899150 non-null  object 
 2   City               899134 non-null  object 
 3   State              899150 non-null  object 
 4   Zip                899164 non-null  int64  
 5   Bank               897605 non-null  object 
 6   BankState          897598 non-null  object 
 7   NAICS              899164 non-null  int64  
 8   ApprovalDate       899164 non-null  object 
 9   ApprovalFY         899164 non-null  object 
 10  Term               899164 non-null  int64  
 11  NoEmp              899164 non-null  int64  
 12  NewExist           899028 non-null  float64
 13  CreateJob          899164 non-null  int64  
 14  RetainedJob        899164 non-null  int64  
 15  FranchiseCode      899164 non-null  int64  
 16  Ur

In [24]:
print(df.isnull().sum())

LoanNr_ChkDgt             0
Name                     14
City                     30
State                    14
Zip                       0
Bank                   1559
BankState              1566
NAICS                     0
ApprovalDate              0
ApprovalFY                0
Term                      0
NoEmp                     0
NewExist                136
CreateJob                 0
RetainedJob               0
FranchiseCode             0
UrbanRural                0
RevLineCr              4528
LowDoc                 2582
ChgOffDate           736465
DisbursementDate       2368
DisbursementGross         0
BalanceGross              0
MIS_Status             1997
ChgOffPrinGr              0
GrAppv                    0
SBA_Appv                  0
dtype: int64


These are the isnull varibles that I will decide to remove. the ChgOfDate is irrelevant in the big picture as it is only for the defaulted loans.

In [25]:
# Data Cleaning 

In [26]:
df = df.dropna(subset=['MIS_Status'])
#subset drops where the speficic cloumn is null instead of just dropna which would drop like everything.
print(df.shape)

(897167, 27)


In [27]:
df = df.dropna(subset=['NewExist'])
print(df.shape)

(897033, 27)


In [28]:
print(df['MIS_Status'].value_counts())

MIS_Status
P I F     739489
CHGOFF    157544
Name: count, dtype: int64


In [29]:
df['default'] = (df['MIS_Status'] == 'CHGOFF').astype(int)

In [30]:
print(df['default'].value_counts())

default
0    739489
1    157544
Name: count, dtype: int64


In [31]:
money_cols = ['GrAppv', 'SBA_Appv']

for col in money_cols:
    df[col] = df[col].str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip().astype(float)

In [32]:
df[['GrAppv', 'SBA_Appv']].dtypes

GrAppv      float64
SBA_Appv    float64
dtype: object

In [33]:
df['sba_guarantee_pct'] = df['SBA_Appv'] / df['GrAppv']

In [34]:
df['naics_sector'] = df['NAICS'].astype(str).str[:2]

In [35]:
df['naics_sector'].value_counts().head(10)

naics_sector
0     201667
44     84560
81     72384
54     67907
72     67494
23     66478
62     55247
42     48664
45     42405
33     38200
Name: count, dtype: int64

In [36]:
sba_clean = df[[
    'State',
    'NAICS',
    'naics_sector',
    'ApprovalFY',
    'Term',
    'NoEmp',
    'NewExist',
    'UrbanRural',
    'GrAppv',
    'SBA_Appv',
    'sba_guarantee_pct',
    'default'
]].copy()

print(sba_clean.shape)
print(sba_clean.head())

(897033, 12)
  State   NAICS naics_sector ApprovalFY  Term  NoEmp  NewExist  UrbanRural  \
0    IN  451120           45       1997    84      4       2.0           0   
1    IN  722410           72       1997    60      2       2.0           0   
2    IN  621210           62       1997   180      7       1.0           0   
3    OK       0            0       1997    60      2       1.0           0   
4    FL       0            0       1997   240     14       1.0           0   

     GrAppv  SBA_Appv  sba_guarantee_pct  default  
0   60000.0   48000.0               0.80        0  
1   40000.0   32000.0               0.80        0  
2  287000.0  215250.0               0.75        0  
3   35000.0   28000.0               0.80        0  
4  229000.0  229000.0               1.00        0  


In [37]:
sba_clean = sba_clean.rename(columns={
    'State': 'state',
    'NAICS': 'naics',
    'naics_sector': 'naics_sector',
    'ApprovalFY': 'approval_fy',
    'Term': 'term',
    'NoEmp': 'no_emp',
    'NewExist': 'new_exist',
    'UrbanRural': 'urban_rural',
    'GrAppv': 'gr_appv',
    'SBA_Appv': 'sba_appv',
    'sba_guarantee_pct': 'sba_guarantee_pct',
    'default': 'default_flag'
})


In [38]:
sba_clean['new_exist'] = sba_clean['new_exist'].astype(int)

In [39]:
print(sba_clean.isnull().sum())

state                13
naics                 0
naics_sector          0
approval_fy           0
term                  0
no_emp                0
new_exist             0
urban_rural           0
gr_appv               0
sba_appv              0
sba_guarantee_pct     0
default_flag          0
dtype: int64


In [40]:
print(sba_clean[sba_clean['state'].isnull()])

       state   naics naics_sector approval_fy  term  no_emp  new_exist  \
49244    NaN       0            0        1966   282       0          0   
264664   NaN       0            0        1987   240      19          1   
306274   NaN  541511           54        1988    73       8          1   
328526   NaN  811192           81        1988   120      17          2   
351072   NaN  532230           53        1989    16       1          2   
366139   NaN  451110           45        1990    84       3          2   
366158   NaN       0            0        1990   204       8          1   
367007   NaN       0            0        1990   240       7          1   
379174   NaN  448310           44        1990    60       4          1   
385418   NaN  532230           53        1990    60       1          1   
869948   NaN  235310           23        1996    60      17          1   
871847   NaN  235610           23        1996    84       3          1   
885335   NaN       0            0     

In [41]:
sba_clean = sba_clean.dropna(subset=['state'])
print(sba_clean.shape)

(897020, 12)


In [42]:
print(sba_clean['approval_fy'].value_counts().tail(10))

approval_fy
1975    11
1976    10
1977    10
1970     8
1983     7
1984     6
1974     4
1969     3
1971     3
1968     1
Name: count, dtype: int64


## Export
Saving the cleaned dataset to data/clean/. Final dataset is 897,020 rows x 12 columns.

In [43]:
sba_clean.to_csv(
    '../data/clean/final_sba_data.csv',
    index=False
)